In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
from tabulate import tabulate
import matplotlib.pyplot as plt

In [ ]:
from google.colab import drive
drive.mount("/content/drive", force_remount=True)

Mounted at /content/drive


In [ ]:
KONDINSKY_PREFIX = "K"
ISHIM_PREFIX = "I"

DATA_DIR = Path("/content/drive/MyDrive/data")
PRIVATE_DATA_DIR = DATA_DIR / "private"

# Загрузка и предобработка данных

Попробуем прочитать исходные данные

Ишимский район

In [ ]:
ishim_table = pd.read_parquet(PRIVATE_DATA_DIR / "ishim.parquet.gzip")

In [ ]:
ishim_table.dtypes

Orders_ID                   int64
RequestId                   int64
SampleId                    int64
LaboratoryId              float64
OrganizationId              int64
BiomaterialId               int64
DateTimeSampling           object
CountBiomaterial            int64
NameUnit                  float64
PatientId                  object
Surname                    object
Name_                      object
Patronymic                 object
Birthday                   object
Sex                       float64
HospitalDepartmentId       object
HospitalDepartmentName     object
DoctorId                  float64
DoctorFullName             object
DiagnosisCode              object
TestId                     object
Mnemonic                   object
TestName                   object
QuantResult                object
QualResult                 object
UnitName                   object
BoundRefer                 object
Comment                    object
DateTime                   object
dtype: object

Кондинский район

In [ ]:
files_for_kondinsky = (PRIVATE_DATA_DIR / "kondinsky").glob("**/*.parquet.gzip")
dfs_for_kondinsky = []
for file in files_for_kondinsky:
  df = pd.read_parquet(file)
  dfs_for_kondinsky.append(df)
kondinsky_table = pd.concat(dfs_for_kondinsky, ignore_index=True)

In [ ]:
kondinsky_table

,№ случая,Лок. Номер,Дата рождения,Дата направления,Пол,Национальность,№ анализа,Наименование анализа,№ теста,Наименование теста,Отделение,Предварительный диагноз,Окончательный диагноз,Результат,№ карты,Тип
0,586605.0,2262942,1984-10-08,2019-04-01 12:21:36,Ж,ТАТАРЫ,37,Исследование физических свойств мокроты,325.0,Пл. эпителий,Поликлиника,J06.8,J06.8,NaN,NaN,NaN
1,585149.0,2265997,1984-03-10,2019-04-01 12:58:22,Ж,РУССКИЕ,313,определение основных групп по системе АВо,450.0,0 lb,Поликлиника,Z32.0,Z32.1,NaN,NaN,NaN
2,581799.0,2233775,2011-01-14,2019-03-25 13:18:20,Ж,РУССКИЕ,313,определение основных групп по системе АВо,453.0,B l,Детская поликлиника,G91.0,G91.0,NaN,NaN,NaN
3,587850.0,1811216,1984-01-08,2019-04-03 09:14:52,Ж,РУССКИЕ,313,определение основных групп по системе АВо,450.0,0 lb,Поликлиника,N84.0,N84.0,NaN,NaN,NaN
4,587862.0,2261889,1975-11-06,2019-04-03 09:24:41,Ж,ТАДЖИКИ,313,определение основных групп по системе АВо,452.0,A b,Поликлиника,N85.0,N85.0,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
594289,NaN,1810244,1953-01-04,2019-06-14 08:09:11,Ж,РУССКИЕ,12,Анализ мочи общий\n,NaN,Ph,Дневной стационар,E11.7,E11.7,5.0,725.0,7.0
594290,NaN,1810244,1953-01-04,2019-06-14 08:09:11,Ж,РУССКИЕ,12,Анализ мочи общий\n,NaN,Удельный вес,Терапевтическое отделение_стац,E11.7,E11.7,1015.0,1652.0,7.0
594291,NaN,1836646,1947-01-15,2019-06-13 00:00:00,Ж,РУССКИЕ,225,ИССЛЕДОВАНИЕ УРОВНЯ МИКРОАЛЬБУМИНА МОЧИ,NaN,АЛЬБ,Терапевтическое отделение_стац,E11.7,E11.7,10.0,3932.0,NaN
594292,NaN,1836646,1947-01-15,2019-06-13 00:00:00,Ж,РУССКИЕ,225,ИССЛЕДОВАНИЕ УРОВНЯ МИКРОАЛЬБУМИНА МОЧИ,NaN,АЛЬБ,Терапевтическое отделение_стац,E11.7,E11.7,10.0,3907.0,NaN


In [ ]:
kondinsky_table.dtypes

№ случая                          float64
Лок. Номер                          int64
Дата рождения              datetime64[us]
Дата направления           datetime64[us]
Пол                                object
Национальность                     object
№ анализа                           int64
Наименование анализа               object
№ теста                           float64
Наименование теста                 object
Отделение                          object
Предварительный диагноз            object
Окончательный диагноз              object
Результат                         float64
№ карты                           float64
Тип                               float64
dtype: object

In [ ]:
kondinsky_rename_dict = {
    "№ случая": "referral_id",
    "Лок. Номер": "patient_id",
    "Дата рождения": "birthday",
    "Пол": "sex",
    "Предварительный диагноз": "diagnosis_code",
    "Наименование теста": "test_name",
    "Результат": "result",
    # Вероятней всего это не дата сдачи анализа,
    # а дата создания направления (чаще всего это может быть дата приема пациента).
    # Однако для простоты можем использовать эту колонку как дату сдачи анализа
    "Дата направления": "sampling_date",
    # "Национальность": "",
    # "№ анализа": "",
    # "Наименование анализа": "",
    # "№ теста": "",
    # "Отделение": "",
    # "Окончательный диагноз": "",
    # "№ карты": "",
    # "Тип": "",
}

ishim_rename_dict = {
    "Orders_ID": "referral_id",
    "PatientId": "patient_id",
    "Birthday": "birthday",
    "Sex": "sex",
    "DiagnosisCode": "diagnosis_code",
    "TestName": "test_name",
    "QuantResult": "result",
    "DateTimeSampling": "sampling_date",
    # "DateTime": "",
    # "RequestId": "",
    # "SampleId": "",
    # "LaboratoryId": "",
    # "OrganizationId": "",
    # "BiomaterialId": "",
    # "CountBiomaterial": "",
    # "NameUnit": "",
    # "Surname": "",
    # "Name_": "",
    # "Patronymic": "",
    # "HospitalDepartmentId": "",
    # "HospitalDepartmentName": "",
    # "DoctorId": "",
    # "DoctorFullName": "",
    # "TestId": "",
    # "Mnemonic": "",
    # "QualResult": "",
    # "UnitName": "",
    # "BoundRefer": "",
    # "Comment": "",
}


def load_kondinsky():
    files = (PRIVATE_DATA_DIR / "kondinsky").glob("**/*.parquet.gzip")
    dfs = []
    for file in files:
        df = pd.read_parquet(file)
        dfs.append(df)
    return pd.concat(dfs, ignore_index=True)


def load_ishim():
    return pd.read_parquet(PRIVATE_DATA_DIR / "ishim.parquet.gzip")


def load_reference():
    ref_file_path = DATA_DIR / "analysis-reference-new.xlsx"

    oak_ref = pd.read_excel(ref_file_path, sheet_name="Справочник ОАК")
    bak_ref = pd.read_excel(ref_file_path, sheet_name="Справочник БАК")

    oak_ref["ref_analysis_name"] = "Общий анализ крови"
    bak_ref["ref_analysis_name"] = "Биохимический анализ крови"

    ref = pd.concat([oak_ref, bak_ref], ignore_index=True)

    return ref


def ishim_date_converter(column):
    return pd.to_datetime(column, format="%d.%m.%Y", errors="coerce")


def kondinsky_date_converter(column):
    return pd.to_datetime(column, dayfirst=True, errors="coerce")


# Используется для предобработки данных в конкретном районе.
# Особенности предобработки задаются через параметры
def preprocess_district_data(
    df,
    rename_dict,
    reference,
    column_in_ref,
    id_prefix,
    date_converter,
    district_name,
):
    # Удаляем ненужные столбцы и переименовываем оставшиеся
    columns_to_save = rename_dict.keys()
    columns_to_drop = set(df.columns).difference(columns_to_save)
    df.drop(columns=columns_to_drop, inplace=True)
    df.rename(columns=rename_dict, inplace=True)

    # Сразу уберем много записей, чтобы не делать ненужные join'ы (ускорит процесс)
    df.dropna(subset=["result", "referral_id"], inplace=True)

    # Заменяем название тестов на справочные
    ref_cols = [
        column_in_ref,
        "ref_analysis_name",
        "ref_test_name",
        "ref_test_mnemonic",
    ]
    df = pd.merge(df, reference[ref_cols], left_on="test_name", right_on=column_in_ref)

    # Убираем колонки left_on и right_on
    df.drop(columns=["test_name", column_in_ref], inplace=True)

    # Создаем уникальные идентификаторы, чтобы случайно не было повторов
    df["referral_id"] = id_prefix + "_" + df["referral_id"].astype(str)
    df["patient_id"] = id_prefix + "_" + df["patient_id"].astype(str)

    # Не можем конвертировать дату в объединенном датасете,
    # потому что один из форматов всегда будет считаться ошибкой. Поэтому делаем здесь,
    # с использованием внешней функции
    df["sampling_date"] = date_converter(df["sampling_date"])
    df["birthday"] = date_converter(df["birthday"])

    # Поможет различать соединенные выборки
    df["district_name"] = district_name

    return df


def preprocess_united_data_inplace(df):
    # errors='coerce' запишет NA, если функция не смогла преобразовать значение

    # Стандартизируем пол
    sex_map = {1.0: "м", 2.0: "ж", "М": "м", "Ж": "ж"}
    df["sex"] = df["sex"].map(sex_map)

    # Стандартизируем разделитель
    result = df["result"].astype("string").str.replace(",", ".")
    df["result"] = pd.to_numeric(result, errors="coerce").astype(float)

    # Посчитаем возраст код на момент сдачи
    df["age_when_sampling"] = pd.to_datetime(df["sampling_date"]).dt.year - pd.to_datetime(df["birthday"]).dt.year


    # Теперь удаляем строки, в которых что-то было где-то неправильно
    columns_to_ignore_dropna = ["ref_test_mnemonic"]
    columns_to_dropna = set(df.columns).difference(columns_to_ignore_dropna)
    df.dropna(subset=columns_to_dropna, inplace=True)

Теперь начинаем работать

In [ ]:
kondinsky_rename_dict = {
    "№ случая": "referral_id",
    "Лок. Номер": "patient_id",
    "Дата рождения": "birthday",
    "Пол": "sex",
    "Предварительный диагноз": "diagnosis_code",
    "Наименование теста": "test_name",
    "Результат": "result",
    # Вероятней всего это не дата сдачи анализа,
    # а дата создания направления (чаще всего это может быть дата приема пациента).
    # Однако для простоты можем использовать эту колонку как дату сдачи анализа
    "Дата направления": "sampling_date",
    # "Национальность": "",
    # "№ анализа": "",
    # "Наименование анализа": "",
    # "№ теста": "",
    # "Отделение": "",
    # "Окончательный диагноз": "",
    # "№ карты": "",
    # "Тип": "",
}

ishim_rename_dict = {
    "Orders_ID": "referral_id",
    "PatientId": "patient_id",
    "Birthday": "birthday",
    "Sex": "sex",
    "DiagnosisCode": "diagnosis_code",
    "TestName": "test_name",
    "QuantResult": "result",
    "DateTimeSampling": "sampling_date",
    # "DateTime": "",
    # "RequestId": "",
    # "SampleId": "",
    # "LaboratoryId": "",
    # "OrganizationId": "",
    # "BiomaterialId": "",
    # "CountBiomaterial": "",
    # "NameUnit": "",
    # "Surname": "",
    # "Name_": "",
    # "Patronymic": "",
    # "HospitalDepartmentId": "",
    # "HospitalDepartmentName": "",
    # "DoctorId": "",
    # "DoctorFullName": "",
    # "TestId": "",
    # "Mnemonic": "",
    # "QualResult": "",
    # "UnitName": "",
    # "BoundRefer": "",
    # "Comment": "",
}


def load_kondinsky():
    files = (PRIVATE_DATA_DIR / "kondinsky").glob("**/*.parquet.gzip")
    dfs = []
    for file in files:
        df = pd.read_parquet(file)
        dfs.append(df)
    return pd.concat(dfs, ignore_index=True)


def load_ishim():
    return pd.read_parquet(PRIVATE_DATA_DIR / "ishim.parquet.gzip")


def load_reference():
    ref_file_path = DATA_DIR / "analysis-reference-new.xlsx"

    oak_ref = pd.read_excel(ref_file_path, sheet_name="Справочник ОАК")
    bak_ref = pd.read_excel(ref_file_path, sheet_name="Справочник БАК")

    oak_ref["ref_analysis_name"] = "Общий анализ крови"
    bak_ref["ref_analysis_name"] = "Биохимический анализ крови"

    ref = pd.concat([oak_ref, bak_ref], ignore_index=True)

    return ref


def ishim_date_converter(column):
    return pd.to_datetime(column, format="%d.%m.%Y", errors="coerce")


def kondinsky_date_converter(column):
    return pd.to_datetime(column, dayfirst=True, errors="coerce")


# Используется для предобработки данных в конкретном районе.
# Особенности предобработки задаются через параметры
def preprocess_district_data(
    df,
    rename_dict,
    reference,
    column_in_ref,
    id_prefix,
    date_converter,
    district_name,
):
    # Удаляем ненужные столбцы и переименовываем оставшиеся
    columns_to_save = rename_dict.keys()
    columns_to_drop = set(df.columns).difference(columns_to_save)
    df.drop(columns=columns_to_drop, inplace=True)
    df.rename(columns=rename_dict, inplace=True)

    # Сразу уберем много записей, чтобы не делать ненужные join'ы (ускорит процесс)
    df.dropna(subset=["result", "referral_id"], inplace=True)

    # Заменяем название тестов на справочные
    ref_cols = [
        column_in_ref,
        "ref_analysis_name",
        "ref_test_name",
        "ref_test_mnemonic",
    ]
    df = pd.merge(df, reference[ref_cols], left_on="test_name", right_on=column_in_ref)

    # Убираем колонки left_on и right_on
    df.drop(columns=["test_name", column_in_ref], inplace=True)

    # Создаем уникальные идентификаторы, чтобы случайно не было повторов
    df["referral_id"] = id_prefix + "_" + df["referral_id"].astype(str)
    df["patient_id"] = id_prefix + "_" + df["patient_id"].astype(str)

    # Не можем конвертировать дату в объединенном датасете,
    # потому что один из форматов всегда будет считаться ошибкой. Поэтому делаем здесь,
    # с использованием внешней функции
    df["sampling_date"] = date_converter(df["sampling_date"])
    df["birthday"] = date_converter(df["birthday"])

    # Поможет различать соединенные выборки
    df["district_name"] = district_name

    return df


def preprocess_united_data_inplace(df):
    # errors='coerce' запишет NA, если функция не смогла преобразовать значение

    # Стандартизируем пол
    sex_map = {1.0: "м", 2.0: "ж", "М": "м", "Ж": "ж"}
    df["sex"] = df["sex"].map(sex_map)

    # Стандартизируем разделитель
    result = df["result"].astype("string").str.replace(",", ".")
    df["result"] = pd.to_numeric(result, errors="coerce").astype(float)

    # Посчитаем возраст код на момент сдачи
    df["age_when_sampling"] = pd.to_datetime(df["sampling_date"]).dt.year - pd.to_datetime(df["birthday"]).dt.year


    # Теперь удаляем строки, в которых что-то было где-то неправильно
    columns_to_ignore_dropna = ["ref_test_mnemonic"]
    columns_to_dropna = set(df.columns).difference(columns_to_ignore_dropna)
    df.dropna(subset=columns_to_dropna, inplace=True)

In [ ]:
print("Loading analysis reference... ", end="", flush=True)
reference = load_reference()
print("done")

print("Loading data [1/2] (kondinsky)... ", end="", flush=True)
df_kondinsky = load_kondinsky()
df_kondinsky = preprocess_district_data(
    df_kondinsky,
    kondinsky_rename_dict,
    reference,
    "kondinsky_both_test_name",
    KONDINSKY_PREFIX,
    kondinsky_date_converter,
    "Кондинский район",
)
print("done")

print("Loading data [2/2] (ishim)... ", end="", flush=True)
df_ishim = load_ishim()
df_ishim = preprocess_district_data(
    df_ishim,
    ishim_rename_dict,
    reference,
    "ishim_test_name",
    ISHIM_PREFIX,
    ishim_date_converter,
    "Ишимский район",
)
print("done")

print("Unioning data... ", end="", flush=True)
df_all = pd.concat([df_kondinsky, df_ishim], ignore_index=True)
print("done")

print("Preprocessing united data... ", end="", flush=True)
preprocess_united_data_inplace(df_all)
print("done")

Loading analysis reference... done
Loading data [1/2] (kondinsky)... done
Loading data [2/2] (ishim)... done
Unioning data... done
Preprocessing united data... done


In [ ]:
# Описание колонок:
#
# referral_id - номер направления
# patient_id - номер пациента
#
# birthday - дата рождения пациента
# sampling_date - дата взятия анализа
# sex - пол
# age_when_sampling - возраст пациента во время взятия анализа (в целых годах; рассчитан на основе даты рождения и даты взятия анализа)
# diagnosis_code - код диагноза (МКБ-10)
#
# result - результат анализа
# ref_analysis_name - название группы анализов (либо общий анализ крови, либо биохимический анализ крови)
# ref_test_name - название конкретного анализа
# ref_test_mnemonic - сокращенное название конкретного анализа (есть не для всех)
#
# district_name - название района
df_all

,referral_id,patient_id,birthday,sampling_date,sex,diagnosis_code,result,ref_analysis_name,ref_test_name,ref_test_mnemonic,district_name,age_when_sampling
0,K_586553.0,K_2073480,1963-03-07 00:00:00,2019-03-29 17:12:53,ж,K29.3,15.0,Биохимический анализ крови,Гаммаглютамилтрансфераза,ГГТ,Кондинский район,56.0
1,K_591244.0,K_1804016,1961-06-29 00:00:00,2019-04-11 08:11:57,ж,C81.0,219.0,Биохимический анализ крови,Гаммаглютамилтрансфераза,ГГТ,Кондинский район,58.0
2,K_593706.0,K_1771783,1959-04-21 00:00:00,2019-04-16 11:09:59,ж,K81.1,49.6,Биохимический анализ крови,Гаммаглютамилтрансфераза,ГГТ,Кондинский район,60.0
3,K_588317.0,K_2303851,2018-02-27 00:00:00,2019-04-05 14:10:47,ж,Q41.1,12.0,Биохимический анализ крови,Гаммаглютамилтрансфераза,ГГТ,Кондинский район,1.0
4,K_588134.0,K_2266120,1945-07-21 00:00:00,2019-04-03 15:13:36,ж,R10.1,17.0,Биохимический анализ крови,Гаммаглютамилтрансфераза,ГГТ,Кондинский район,74.0
...,...,...,...,...,...,...,...,...,...,...,...,...
772618,I_100166167,I_720011_22606,1939-08-10 00:00:00,2019-06-13 00:00:00,м,K40.9,106.1,Биохимический анализ крови,Хлориды,NaN,Ишимский район,80.0
772619,I_100166231,I_720011_60200,1984-08-01 00:00:00,2019-06-13 00:00:00,ж,Z34.0,104.5,Биохимический анализ крови,Хлориды,NaN,Ишимский район,35.0
772620,I_100166013,I_720011_134213,1939-07-06 00:00:00,2019-06-13 00:00:00,м,S42.20,101.7,Биохимический анализ крови,Хлориды,NaN,Ишимский район,80.0
772621,I_100166266,I_720011_217532,1952-06-13 00:00:00,2019-06-13 00:00:00,м,I21.0,100.1,Биохимический анализ крови,Хлориды,NaN,Ишимский район,67.0


In [ ]:
# Пропущенные значения есть только для мнемоник, во всех других местах все значения есть
df_all.isna().sum()

referral_id               0
patient_id                0
birthday                  0
sampling_date             0
sex                       0
diagnosis_code            0
result                    0
ref_analysis_name         0
ref_test_name             0
ref_test_mnemonic    176684
district_name             0
age_when_sampling         0
dtype: int64

In [ ]:
# Для всех колонок есть указаны типы данных
df_all.dtypes

referral_id           object
patient_id            object
birthday              object
sampling_date         object
sex                   object
diagnosis_code        object
result               float64
ref_analysis_name     object
ref_test_name         object
ref_test_mnemonic     object
district_name         object
age_when_sampling    float64
dtype: object

## Удаляем строки с лишними и пропущенными данными

### Удаляем дублирующиеся данные

In [ ]:
# смотрим, есть ли дублирующиеся данные
df_all.duplicated()

0         False
1         False
2         False
3         False
4         False
          ...  
772618    False
772619    False
772620    False
772621    False
772622    False
Length: 700951, dtype: bool

In [ ]:
# но не факт, что дубликатов нет, поэтому их удалим
df_all = df_all.drop_duplicates()

In [ ]:
# проверяем количество строк в таблице
len(df_all)

682237

Мы удалили дубликаты данных. Количество строк стало меньше.

### Удаляем пропущенные данные

In [ ]:
# смотрим, есть ли пропущенные данные
df_all.isna()

,referral_id,patient_id,birthday,sampling_date,sex,diagnosis_code,result,ref_analysis_name,ref_test_name,ref_test_mnemonic,district_name,age_when_sampling
0,False,False,False,False,False,False,False,False,False,False,False,False
1,False,False,False,False,False,False,False,False,False,False,False,False
2,False,False,False,False,False,False,False,False,False,False,False,False
3,False,False,False,False,False,False,False,False,False,False,False,False
4,False,False,False,False,False,False,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...
772618,False,False,False,False,False,False,False,False,False,True,False,False
772619,False,False,False,False,False,False,False,False,False,True,False,False
772620,False,False,False,False,False,False,False,False,False,True,False,False
772621,False,False,False,False,False,False,False,False,False,True,False,False


In [ ]:
# смотрим, есть ли пропущенные данные
df_all.isnull()

,referral_id,patient_id,birthday,sampling_date,sex,diagnosis_code,result,ref_analysis_name,ref_test_name,ref_test_mnemonic,district_name,age_when_sampling
0,False,False,False,False,False,False,False,False,False,False,False,False
1,False,False,False,False,False,False,False,False,False,False,False,False
2,False,False,False,False,False,False,False,False,False,False,False,False
3,False,False,False,False,False,False,False,False,False,False,False,False
4,False,False,False,False,False,False,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...
772618,False,False,False,False,False,False,False,False,False,True,False,False
772619,False,False,False,False,False,False,False,False,False,True,False,False
772620,False,False,False,False,False,False,False,False,False,True,False,False
772621,False,False,False,False,False,False,False,False,False,True,False,False


In [ ]:
# удаляем данные
df_all = df_all.dropna()

In [ ]:
df_all

,referral_id,patient_id,birthday,sampling_date,sex,diagnosis_code,result,ref_analysis_name,ref_test_name,ref_test_mnemonic,district_name,age_when_sampling
0,K_586553.0,K_2073480,1963-03-07 00:00:00,2019-03-29 17:12:53,ж,K29.3,15.0,Биохимический анализ крови,Гаммаглютамилтрансфераза,ГГТ,Кондинский район,56.0
1,K_591244.0,K_1804016,1961-06-29 00:00:00,2019-04-11 08:11:57,ж,C81.0,219.0,Биохимический анализ крови,Гаммаглютамилтрансфераза,ГГТ,Кондинский район,58.0
2,K_593706.0,K_1771783,1959-04-21 00:00:00,2019-04-16 11:09:59,ж,K81.1,49.6,Биохимический анализ крови,Гаммаглютамилтрансфераза,ГГТ,Кондинский район,60.0
3,K_588317.0,K_2303851,2018-02-27 00:00:00,2019-04-05 14:10:47,ж,Q41.1,12.0,Биохимический анализ крови,Гаммаглютамилтрансфераза,ГГТ,Кондинский район,1.0
4,K_588134.0,K_2266120,1945-07-21 00:00:00,2019-04-03 15:13:36,ж,R10.1,17.0,Биохимический анализ крови,Гаммаглютамилтрансфераза,ГГТ,Кондинский район,74.0
...,...,...,...,...,...,...,...,...,...,...,...,...
735129,I_100166013,I_720011_134213,1939-07-06 00:00:00,2019-06-13 00:00:00,м,S42.20,18.8,Биохимический анализ крови,Аспартатаминотрансфераза,АСТ,Ишимский район,80.0
735130,I_100166196,I_720011_140822,1954-01-08 00:00:00,2019-06-13 00:00:00,ж,K76.9,42.4,Биохимический анализ крови,Аспартатаминотрансфераза,АСТ,Ишимский район,65.0
735131,I_100166276,I_720011_217523,1935-11-01 00:00:00,2019-06-13 00:00:00,ж,N19,19.1,Биохимический анализ крови,Аспартатаминотрансфераза,АСТ,Ишимский район,84.0
735132,I_100165370,I_720011_61003,1984-03-13 00:00:00,2019-06-13 00:00:00,ж,M42.1,42.1,Биохимический анализ крови,Аспартатаминотрансфераза,АСТ,Ишимский район,35.0


# Работа с данными

<h2>Выведем параметры общего и биохимического анализов крови</h2>

<h2>Отфильтруем таблицу по параметрам: диагнозам</h2>

In [ ]:
df_hypertension = df_all.loc[(df_all['diagnosis_code']=='I10')|(df_all['diagnosis_code']=='I11.0')|(df_all['diagnosis_code']=='I11.9')|(df_all['diagnosis_code']=='I12.0')|(df_all['diagnosis_code']=='I12.9')|(df_all['diagnosis_code']=='I13.0')|(df_all['diagnosis_code']=='I13.1')|(df_all['diagnosis_code']=='I13.2')|(df_all['diagnosis_code']=='I13.9')|(df_all['diagnosis_code']=='I15')]

In [ ]:
df_hypertension

,referral_id,patient_id,birthday,sampling_date,sex,diagnosis_code,result,ref_analysis_name,ref_test_name,ref_test_mnemonic,district_name,age_when_sampling
389,K_593623.0,K_1808669,1946-09-10 00:00:00,2019-04-17 00:00:00,м,I11.9,44.0,Общий анализ крови,Гематокрит,HCT,Кондинский район,73.0
443,K_582659.0,K_1814188,1950-05-04 00:00:00,2019-04-16 16:19:42,м,I11.9,40.5,Общий анализ крови,Гематокрит,HCT,Кондинский район,69.0
444,K_595490.0,K_2300967,1950-04-27 00:00:00,2019-04-19 09:42:52,ж,I11.9,32.0,Общий анализ крови,Гематокрит,HCT,Кондинский район,69.0
449,K_598100.0,K_1828108,1984-07-11 00:00:00,2019-04-25 10:53:01,ж,I11.9,38.5,Общий анализ крови,Гематокрит,HCT,Кондинский район,35.0
459,K_597814.0,K_753251,1972-07-14 00:00:00,2019-04-24 17:51:12,м,I11.9,43.9,Общий анализ крови,Гематокрит,HCT,Кондинский район,47.0
...,...,...,...,...,...,...,...,...,...,...,...,...
735067,I_100165834,I_720011_5366,1954-01-07 00:00:00,2019-06-13 00:00:00,ж,I10,30.1,Биохимический анализ крови,Аспартатаминотрансфераза,АСТ,Ишимский район,65.0
735083,I_100165806,I_720011_128758,1954-12-25 00:00:00,2019-06-13 00:00:00,ж,I11.9,19.2,Биохимический анализ крови,Аспартатаминотрансфераза,АСТ,Ишимский район,65.0
735090,I_100165393,I_720011_171076,1954-10-02 00:00:00,2019-06-13 00:00:00,ж,I11.9,23.0,Биохимический анализ крови,Аспартатаминотрансфераза,АСТ,Ишимский район,65.0
735093,I_100165218,I_720011_112001,1948-04-26 00:00:00,2019-06-13 00:00:00,ж,I10,23.2,Биохимический анализ крови,Аспартатаминотрансфераза,АСТ,Ишимский район,71.0


Выведем список параметров

In [ ]:
df_hypertension.ref_test_name.unique()

array(['Гематокрит', 'Липопротеиды низкой плотности', 'Лимфоциты',
       'Гемоглобин', 'Тромбоциты', 'Скорость оседания эритроцитов',
       'Лейкоциты', 'Моноциты', 'Эритроциты',
       'Липопротеиды высокой плотности', 'Аспартатаминотрансфераза',
       'Аланинаминотрансфераза', 'Базофилы', 'Эозинофилы',
       'Средний объем эритроцита',
       'Среднее содержание гемоглобина в эритроците',
       'Средний объем тромбоцитов', 'Нейтрофилы',
       'Гаммаглютамилтрансфераза'], dtype=object)

In [ ]:
df_hypertension.ref_test_mnemonic.unique()

array(['HCT', 'LDL', 'LYM', 'HGB', 'PLT', 'ESR', 'WBC', 'MONO', 'RBC',
       'HDL', 'АСТ', 'АЛТ', 'BASO', 'EOS', 'MCV', 'MCH', 'MPV', 'NEU',
       'ГГТ'], dtype=object)

<h3>Выводим исходные данные</h3>

<h5>Делаем словарь для вводных данных</h5>

In [ ]:
input_data_descript = {
    'ESR': '- скорости оседания эритроцитов (СОЭ);',
    'RBC': '- количества эритроцитов;',
    'LYM': '- процента лифмоцитов;',
    'PLT': '- количества тромбоцитов;',
    'HGB': '- количества гемоглобина;',
    'АЛТ': '- активности аланинаминотрансферазы (АЛТ);',
    'BASO': '- процента базофилов;',
    'WBC': '- количества лейкоцитов;',
    'MONO': '- процента моноцитов;',
    'HCT': '- гематокрита;',
    'EOS': '- доли эозинофилов;',
    'LDL': "- концентрации липопротеидов низкой плотности (ЛПНП);",
    'HDL': "- концентрации липопротеидов высокой плотности (ЛПВП);",
    'АСТ': '- количества аспартатаминотрансферазы (АСТ);',
    'MCV': '- среднего объема эритроцита;',
    'MCH': '- среднего содержания гемоглобина в эритроците;',
    'MPV': '- среднего объема тромбоцитов;',
    'NEU': '- процента нейтрофилов;',
    'ГГТ': '- активности гаммаглютамилтрансферазы (ГГТ);'
}

<h5>Это список исходных данных</h5>

In [ ]:
input_tests = ['АЛТ', 'LDL', 'HDL',  'АСТ', 'ГГТ']

In [ ]:
print('\033[1mИсходные данные: \033[0m')
print('Оцениваем корреляционная связь между следующими признаками: ') #
# выводим значение для ключа словаря - элемента массива
for test in input_tests:
    for key in input_data_descript:
        if test == key:
            print(input_data_descript[key])

Исходные данные: 
Оцениваем корреляционная связь между следующими признаками: 
- активности аланинаминотрансферазы (АЛТ);
- концентрации липопротеидов низкой плотности (ЛПНП);
- концентрации липопротеидов высокой плотности (ЛПВП);
- количества аспартатаминотрансферазы (АСТ);
- активности гаммаглютамилтрансферазы (ГГТ);


<h5>Делаем функции для удаления значений</h5>

<h5>Делаем массив со значениями пола</h5>


<h5>Удаляем лишние значения для всех</h5>

In [ ]:
def samples_filler(df_output, parameter):
    df_output.drop(df_output[df_output['ref_test_mnemonic'] == parameter].index)
    df_output.drop(df_output[df_output['result'] >= df_output['result'].mean()+2.8*df_output['result'].std()].index)
    df_output.drop(df_output[df_output['result'] <= df_output['result'].mean()-2.8*df_output['result'].std()].index)
    return df_output[['patient_id', 'sampling_date', 'result']].rename(columns = {'result': parameter})

<h5>Записываем выборки в список</h5>

In [ ]:
# делаем список для выборок
samples = []
# добавляем в массив значение выборки
for test in input_tests:
    samples.append(samples_filler(df_hypertension, test))


In [ ]:
samples

[             patient_id        sampling_date   АЛТ
 389           K_1808669  2019-04-17 00:00:00  44.0
 443           K_1814188  2019-04-16 16:19:42  40.5
 444           K_2300967  2019-04-19 09:42:52  32.0
 449           K_1828108  2019-04-25 10:53:01  38.5
 459            K_753251  2019-04-24 17:51:12  43.9
 ...                 ...                  ...   ...
 735067    I_720011_5366  2019-06-13 00:00:00  30.1
 735083  I_720011_128758  2019-06-13 00:00:00  19.2
 735090  I_720011_171076  2019-06-13 00:00:00  23.0
 735093  I_720011_112001  2019-06-13 00:00:00  23.2
 735107   I_720011_23351  2019-06-13 00:00:00  19.4
 
 [73099 rows x 3 columns],
              patient_id        sampling_date   LDL
 389           K_1808669  2019-04-17 00:00:00  44.0
 443           K_1814188  2019-04-16 16:19:42  40.5
 444           K_2300967  2019-04-19 09:42:52  32.0
 449           K_1828108  2019-04-25 10:53:01  38.5
 459            K_753251  2019-04-24 17:51:12  43.9
 ...                 ...           

<h5>Соединим таблицы</h5>

In [ ]:
# итоговую таблицу будем делать из первой
table_final = samples[0]
# соединяем таблицы
for i in range(1, 5):
    table_final = pd.merge(samples[i], table_final, on = ['patient_id', 'sampling_date'])

Попробуем сделать матрицы корреляции

Матрицы корреляции по всем пациентам

In [ ]:
table_final

In [ ]:
table_final = table_final.drop(columns = ['patient_id', 'sampling_date'])

In [ ]:
table_final_corr = table_final.corr()

In [ ]:
# делаем тепловую карту
import seaborn as sns
heat_map_all = sns.heatmap(table_final_corr, annot = table_final_corr, fmt = '.2f', vmax = 1, vmin = -1)
plt.title('Тепловая карта корреляции (Все пациенты)')

<h5>Сделаем таблицы по отдельным полам</h5>

In [ ]:
# функция формирования выборок для каждого пола
def samples_filler_sex(df_output, parameter, sex):
    df_output.drop(df_output[df_output['sex'] == sex].index)
    df_output.drop(df_output[df_output['ref_test_mnemonic'] == parameter].index)
    df_output.drop(df_output[df_output['result'] >= df_output['result'].mean()+2.8*df_output['result'].std()].index)
    df_output.drop(df_output[df_output['result'] <= df_output['result'].mean()-2.8*df_output['result'].std()].index)
    return df_output[['patient_id', 'sampling_date', 'result']].rename(columns = {'result': parameter})

Тепловая карта по мужчинам

In [ ]:
# делаем массив для выборок
samples_men = []
# добавляем в массив значение выборки
for test in input_tests:
    samples_men.append(samples_filler_sex(df_hypertension, test, 'м'))

Проверяем

In [ ]:
samples_men

Матрицы корреляции по пациентам-мужчинам

In [ ]:
# итоговую таблицу будем делать из первой
table_final_men = samples_men[0]
# соединяем таблицы
for i in range(1, 3):
    table_final_men = pd.merge(samples_men[i], table_final_men, on = ['patient_id', 'sampling_date'])

In [ ]:
table_final_men = table_final_men.drop(columns = ['patient_id', 'sampling_date'])

In [ ]:
table_final_men_corr = table_final_men.corr()

In [ ]:
# делаем тепловую карту
import seaborn as sns
heat_map_men = sns.heatmap(table_final_men_corr, annot = table_final_men_corr, fmt = '.2f', vmax = 1, vmin = -1)
plt.title('Тепловая карта корреляции (Мужчины)')

Тепловая карта по женщинам

In [ ]:
# делаем массив для выборок
samples_women = []
# добавляем в массив значение выборки
for test in input_tests:
    samples_women.append(samples_filler_sex(df_hypertension, test, 'ж'))

Проверяем

In [ ]:
samples_women

Матрицы корреляции по пациентам-женщинам

In [ ]:
# итоговую таблицу будем делать из первой
table_final_women = samples_women[0]
# соединяем таблицы
for i in range(1, 3):
    table_final_women = pd.merge(samples_women[i], table_final_women, on = ['patient_id', 'sampling_date'])

In [ ]:
table_final_women = table_final_women.drop(columns = ['patient_id', 'sampling_date'])

In [ ]:
table_final_women_corr = table_final_women.corr()

In [ ]:
# делаем тепловую карту
import seaborn as sns
heat_map_women = sns.heatmap(table_final_women_corr, annot = table_final_women_corr, fmt = '.2f', vmax = 1, vmin = -1)
plt.title('Тепловая карта корреляции (Женщины)')

<h3>Попробуем извлечь признаки из матрицы корреляции</h3>



In [ ]:
# попробуем сделать словарь из ключей
# для этого есть функция
import operator
import scipy
def corr_dict_generation(biochem_total, method):
    corr_dict = {}
    for el1 in list(biochem_total):
        for el2 in list(biochem_total):
            if el1 != el2:
                corr_dict[str(el1)+'-'+str(el2)] = abs(biochem_total[el1].corr(biochem_total[el2], method = method))# надо добавить коэффициент корреляции
    corr_dict_2 = sorted(corr_dict.items(), key = operator.itemgetter(1), reverse = True)
    corr_dict_3 = [corr_dict_2[0], corr_dict_2[2]]
    return corr_dict_3

Выведем рейтинг по всем пациентам

In [ ]:
# функция для карты с методом Пирсона
text_for_table = corr_dict_generation(table_final_corr, 'pearson')
text_for_table

Выведем рейтинг по мужчинам

In [ ]:
# функция для карты с методом Пирсона
text_for_table_men = corr_dict_generation(table_final_men_corr, 'pearson')
text_for_table_men

Выведем рейтинг по женщинам

In [ ]:
# функция для карты с методом Пирсона
text_for_table_women = corr_dict_generation(table_final_women_corr, 'pearson')
text_for_table_women

<h3>Таблицы по полам</h3>

<h5>Вывод таблицы</h5>

In [ ]:
table_sex = [['Пара признаков для всех пациентов', 'Пара признаков для мужчин', 'Пара признаков для женщин'],
 [str(text_for_table[0][0])+' (r = '+('{:.2f}'.format(text_for_table[0][1]))+')', str(text_for_table_men[0][0])+' (r = '+('{:.2f}'.format(text_for_table_men[0][1]))+')', str(text_for_table_women[0][0])+' (r = '+('{:.2f}'.format(text_for_table_women[0][1]))+')'],
[str(text_for_table[1][0])+' (r = '+('{:.2f}'.format(text_for_table[1][1]))+')', str(text_for_table_men[1][0])+' (r = '+('{:.2f}'.format(text_for_table_men[1][1]))+')', str(text_for_table_women[1][0])+' (r = '+('{:.2f}'.format(text_for_table_women[1][1]))+')'],]


In [ ]:
print('\033[1mI. ОПИСАННИЕ РЕЗУЛЬТАТОВ ДЛЯ ЛИЦ РАЗНОГО ПОЛА\033[0m')
print('\033[1mЗначения коэффициентов корреляции для лиц разного пола\033[0m')
print(tabulate(table_sex))

In [ ]:
print('Использовался коэффициент корреляции Пирсона')

<h3>Описание результатов</h3>

<h5>Вывод текста</h5>

In [ ]:
# печатаем
print('\033[1mВывод по полам\033[0m')
print('\033[1mВсе пациенты\033[0m')
print('Пара признаков с наиболее сильной корреляционной связью - '+str(text_for_table[0][0])+'. Ее коэффициент корреляции равен '+str('{:.2f}'.format(text_for_table[0][1])+'.'))
print('В порядке убывания корреляционной связи находятся: ')
print('- пара признаков '+str(text_for_table[1][0])+'. Ее коэффициент корреляции равен '+str('{:.2f}'.format(text_for_table[1][1])+'.'))
print('\033[1mМужчины\033[0m')
print('Пара признаков с наиболее сильной корреляционной связью - '+str(text_for_table_men[0][0])+'. Ее коэффициент корреляции равен '+str('{:.2f}'.format(text_for_table_men[0][1])+'.'))
print('В порядке убывания корреляционной связи находятся: ')
print('- пара признаков '+str(text_for_table_men[1][0])+'. Ее коэффициент корреляции равен '+str('{:.2f}'.format(text_for_table_men[1][1])+'.'))
print('\033[1mЖенщины\033[0m')
print('Пара признаков с наиболее сильной корреляционной связью - '+str(text_for_table_women[0][0])+'. Ее коэффициент корреляции равен '+str('{:.2f}'.format(text_for_table_women[0][1])+'.'))
print('В порядке убывания корреляционной связи находятся: ')
print('- пара признаков '+str(text_for_table_women[1][0])+'. Ее коэффициент корреляции равен '+str('{:.2f}'.format(text_for_table_women[1][1])+'.'))


<h3>Таблицы по возрастам</h3>

<h2>Тепловая карта по пациентам до 50 лет</h2>

In [ ]:
# функция формирования выборок для каждого пола
def samples_filler_less_50(df_output, parameter):
    df_output.drop(df_output[df_output['age_when_sampling'] < 50].index)
    df_output.drop(df_output[df_output['ref_test_mnemonic'] == parameter].index)
    df_output.drop(df_output[df_output['result'] >= df_output['result'].mean()+2.8*df_output['result'].std()].index)
    df_output.drop(df_output[df_output['result'] <= df_output['result'].mean()-2.8*df_output['result'].std()].index)
    return df_output[['patient_id', 'sampling_date', 'result']].rename(columns = {'result': parameter})

In [ ]:
# делаем массив для выборок
samples_less_50 = []
# добавляем в массив значение выборки
for test in input_tests:
    samples_less_50.append(samples_filler_less_50(df_hypertension, test))

In [ ]:
samples_less_50

Матрицы корреляции по пациентам до 50 лет

In [ ]:
# итоговую таблицу будем делать из первой
table_final_less_50 = samples_less_50[0]
# соединяем таблицы
for i in range(1, 3):
    table_final_less_50 = pd.merge(samples_less_50[i], table_final_less_50, on = ['patient_id', 'sampling_date'])

In [ ]:
table_final_less_50 = table_final_less_50.drop(columns = ['patient_id', 'sampling_date'])

In [ ]:
table_final_less_50_corr = table_final_less_50.corr()

In [ ]:
# делаем тепловую карту
import seaborn as sns
heat_map_less_50 = sns.heatmap(table_final_less_50_corr, annot = table_final_less_50_corr, fmt = '.2f', vmax = 1, vmin = -1)
plt.title('Тепловая карта корреляции (Пациенты до 50 лет)')

Выведем рейтинг по пациентам до 50 лет

In [ ]:
# функция для карты с методом Пирсона
text_for_table_less_50 = corr_dict_generation(table_final_less_50_corr, 'pearson')
text_for_table_less_50

<h2>Тепловая карта по пациентам 50-59 лет</h2>

In [ ]:
# функция формирования выборок для каждого пола
def samples_filler_50_and_60(df_output, parameter, age_1, age_2):
    df_output.drop(df_output[df_output['age_when_sampling'] < age_1].index)
    df_output.drop(df_output[df_output['age_when_sampling'] > age_2].index)
    df_output.drop(df_output[df_output['ref_test_mnemonic'] == parameter].index)
    df_output.drop(df_output[df_output['result'] >= df_output['result'].mean()+2.8*df_output['result'].std()].index)
    df_output.drop(df_output[df_output['result'] <= df_output['result'].mean()-2.8*df_output['result'].std()].index)
    return df_output[['patient_id', 'sampling_date', 'result']].rename(columns = {'result': parameter})

In [ ]:
# делаем массив для выборок
samples_50_59 = []
# добавляем в массив значение выборки
for test in input_tests:
    samples_50_59.append(samples_filler_50_and_60(df_hypertension, test, 50, 59))

In [ ]:
samples_50_59

Делаем тепловую карту

In [ ]:
# итоговую таблицу будем делать из первой
table_final_50_59 = samples_50_59[0]
# соединяем таблицы
for i in range(1, 3):
    table_final_50_59 = pd.merge(samples_50_59[i], table_final_50_59, on = ['patient_id', 'sampling_date'])

In [ ]:
table_final_50_59 = table_final_50_59.drop(columns = ['patient_id', 'sampling_date'])

In [ ]:
table_final_50_59_corr = table_final_50_59.corr()

In [ ]:
# делаем тепловую карту
import seaborn as sns
heat_map_50_59 = sns.heatmap(table_final_50_59_corr, annot = table_final_50_59_corr, fmt = '.2f', vmax = 1, vmin = -1)
plt.title('Тепловая карта корреляции (Пациенты 50-59 лет)')

Выведем рейтинг по пациентам 50-59 лет

In [ ]:
# функция для карты с методом Пирсона
text_for_table_50_59 = corr_dict_generation(table_final_50_59_corr, 'pearson')
text_for_table_50_59

<h2>Тепловая карта по пациентам 60-69 лет</h2>

In [ ]:
# делаем массив для выборок
samples_60_69 = []
# добавляем в массив значение выборки
for test in input_tests:
    samples_60_69.append(samples_filler_50_and_60(df_hypertension, test, 60, 69))

In [ ]:
samples_60_69

Делаем тепловую карту

In [ ]:
# итоговую таблицу будем делать из первой
table_final_60_69 = samples_60_69[0]
# соединяем таблицы
for i in range(1, 3):
    table_final_60_69 = pd.merge(samples_60_69[i], table_final_60_69, on = ['patient_id', 'sampling_date'])

In [ ]:
table_final_60_69 = table_final_60_69.drop(columns = ['patient_id', 'sampling_date'])

In [ ]:
table_final_60_69_corr = table_final_60_69.corr()

In [ ]:
# делаем тепловую карту
import seaborn as sns
heat_map_60_69 = sns.heatmap(table_final_60_69_corr, annot = table_final_60_69_corr, fmt = '.2f', vmax = 1, vmin = -1)
plt.title('Тепловая карта корреляции (Пациенты 60-69 лет)')

Выведем рейтинг по пациентам 60-69 лет

In [ ]:
# функция для карты с методом Пирсона
text_for_table_60_69 = corr_dict_generation(table_final_60_69_corr, 'pearson')
text_for_table_60_69

<h2>Тепловая карта по пациентам от 70 лет</h2>

In [ ]:
# функция формирования выборок для каждого пола
def samples_filler_70_and_more(df_output, parameter):
    df_output.drop(df_output[df_output['age_when_sampling'] >= 70].index)
    df_output.drop(df_output[df_output['ref_test_mnemonic'] == parameter].index)
    df_output.drop(df_output[df_output['result'] >= df_output['result'].mean()+2.8*df_output['result'].std()].index)
    df_output.drop(df_output[df_output['result'] <= df_output['result'].mean()-2.8*df_output['result'].std()].index)
    return df_output[['patient_id', 'sampling_date', 'result']].rename(columns = {'result': parameter})

In [ ]:
# делаем массив для выборок
samples_more_70 = []
# добавляем в массив значение выборки
for test in input_tests:
    samples_more_70.append(samples_filler_70_and_more(df_hypertension, test))

Делаем тепловую карту

In [ ]:
# итоговую таблицу будем делать из первой
table_final_70_and_more = samples_more_70[0]
# соединяем таблицы
for i in range(1, 3):
    table_final_70_and_more = pd.merge(samples_more_70[i], table_final_70_and_more, on = ['patient_id', 'sampling_date'])

In [ ]:
table_final_70_and_more = table_final_70_and_more.drop(columns = ['patient_id', 'sampling_date'])

In [ ]:
table_final_70_and_more_corr = table_final_70_and_more.corr()

In [ ]:
# делаем тепловую карту
import seaborn as sns
heat_map_70_and_more = sns.heatmap(table_final_70_and_more_corr, annot = table_final_70_and_more_corr, fmt = '.2f', vmax = 1, vmin = -1)
plt.title('Тепловая карта корреляции (Пациенты от 70 лет)')

Выведем рейтинг по пациентам от 70 лет

In [ ]:
# функция для карты с методом Пирсона
text_for_table_70_and_more = corr_dict_generation(table_final_70_and_more_corr, 'pearson')
text_for_table_70_and_more

<h5>Вывод текста</h5>

In [ ]:
print('\033[1mI. ОПИСАННИЕ РЕЗУЛЬТАТОВ ДЛЯ ЛИЦ РАЗНОГО ВОЗРАСТА\033[0m')
print('\033[1mЗначения коэффициентов корреляции для лиц разного возраста\033[0m')

In [ ]:
table_age = [['Пара признаков для всех пациентов', 'Пара признаков для людей до 50 лет', 'Пара признаков 50-59 лет', 'Пара признаков 60-69 лет', 'Пара признаков для людей от 70 лет'],
  [str(text_for_table[0][0])+' (r = '+('{:.2f}'.format(text_for_table[0][1]))+')', str(text_for_table_less_50[0][0])+' (r = '+('{:.2f}'.format(text_for_table_less_50[0][1]))+')', str(text_for_table_50_59[0][0])+' (r = '+('{:.2f}'.format(text_for_table_50_59[0][1]))+')',  str(text_for_table_60_69[0][0])+' (r = '+('{:.2f}'.format(text_for_table_60_69[0][1]))+')', str(text_for_table_70_and_more[0][0])+' (r = '+('{:.2f}'.format(text_for_table_70_and_more[0][1]))+')'],
[str(text_for_table[1][0])+' (r = '+('{:.2f}'.format(text_for_table[1][1]))+')', str(text_for_table_less_50[1][0])+' (r = '+('{:.2f}'.format(text_for_table_less_50[1][1]))+')', str(text_for_table_50_59[1][0])+' (r = '+('{:.2f}'.format(text_for_table_50_59[1][1]))+')',  str(text_for_table_60_69[1][0])+' (r = '+('{:.2f}'.format(text_for_table_60_69[1][1]))+')',  str(text_for_table_70_and_more[1][0])+' (r = '+('{:.2f}'.format(text_for_table_70_and_more[1][1]))+')']
             ]

In [ ]:
print(tabulate(table_age))

<h3>Описание результатов</h3>

<h5>Вывод текста</h5>

In [ ]:
# печатаем
print('\033[1mВывод по возрастам\033[0m')
print('\033[1mВсе пациенты\033[0m')
print('Пара признаков с наиболее сильной корреляционной связью - '+str(text_for_table[0][0])+'. Ее коэффициент корреляции равен '+str('{:.2f}'.format(text_for_table[0][1])+'.'))
print('В порядке убывания корреляционной связи находятся: ')
print('- пара признаков '+str(text_for_table[1][0])+'. Ее коэффициент корреляции равен '+str('{:.2f}'.format(text_for_table[1][1])+'.'))
print('\033[1mПациенты в возрасте до 50 лет\033[0m')
print('Пара признаков с наиболее сильной корреляционной связью - '+str(text_for_table_less_50[0][0])+'. Ее коэффициент корреляции равен '+str('{:.2f}'.format(text_for_table_less_50[0][1])+'.'))
print('В порядке убывания корреляционной связи находятся: ')
print('- пара признаков '+str(text_for_table_less_50[1][0])+'. Ее коэффициент корреляции равен '+str('{:.2f}'.format(text_for_table_less_50[1][1])+'.'))
print('\033[1mПациенты в возрасте 50-59 лет\033[0m')
print('Пара признаков с наиболее сильной корреляционной связью - '+str(text_for_table_50_59[0][0])+'. Ее коэффициент корреляции равен '+str('{:.2f}'.format(text_for_table_50_59[0][1])+'.'))
print('В порядке убывания корреляционной связи находятся: ')
print('- пара признаков '+str(text_for_table_50_59[1][0])+'. Ее коэффициент корреляции равен '+str('{:.2f}'.format(text_for_table_50_59[1][1])+'.'))
print('\033[1mПациенты в возрасте 60-69 лет\033[0m')
print('Пара признаков с наиболее сильной корреляционной связью - '+str(text_for_table_60_69[0][0])+'. Ее коэффициент корреляции равен '+str('{:.2f}'.format(text_for_table_60_69[0][1])+'.'))
print('В порядке убывания корреляционной связи находятся: ')
print('- пара признаков '+str(text_for_table_60_69[1][0])+'. Ее коэффициент корреляции равен '+str('{:.2f}'.format(text_for_table_60_69[1][1])+'.'))
print('\033[1mПациенты в возрасте от 70 лет\033[0m')
print('Пара признаков с наиболее сильной корреляционной связью - '+str(text_for_table_60_69[0][0])+'. Ее коэффициент корреляции равен '+str('{:.2f}'.format(text_for_table_60_69[0][1])+'.'))
print('В порядке убывания корреляционной связи находятся: ')
print('- пара признаков '+str(text_for_table_70_and_more[1][0])+'. Ее коэффициент корреляции равен '+str('{:.2f}'.format(text_for_table_70_and_more[1][1])+'.'))